# Saadi quickstart

Index one document once, then retrieve and answer as often as you like. The state lives in a flat directory you can inspect.


In [ ]:
%pip install -q "saadi[openai] @ git+https://github.com/ZeinabTaghavi/SAADI.git"


In [ ]:
import os
os.environ.setdefault('OPENAI_API_KEY', 'paste-your-key-here')


In [ ]:
from saadi import Saadi

doc = Saadi(llm='openai:gpt-4.1-mini', state_dir='saadi_state/aurora', chunk_size=100)
report = doc.index('aurora_report/document.txt')
report


## The state the model built

`memory` holds entity records; `history` holds the chunks (level 0) and the fold summaries above them.


In [ ]:
for entity in doc.memory[:8]:
    print(entity.name, '|', ', '.join(entity.aliases), '|', '; '.join(entity.descriptions))


In [ ]:
for item in doc.history.get(1, []):
    print(f'L1 steps {item.start_step}-{item.end_step}:', item.text)


## Retrieve and answer

Retrieval fuses the chunk text, the entity memory, and the summaries in one pass, with no model call. `answer` adds the folded context and one generation call.


In [ ]:
for hit in doc.retrieve('Who keeps the Corvin station?', k=3):
    print(hit.rank, hit.chunk_index, round(hit.score, 3), hit.text[:120])


In [ ]:
reply = doc.answer('Who keeps the Corvin station?')
print(reply.text)


## Extend without rebuilding

Appending text reuses every unchanged chunk from the cache; only the new chunks cost model calls.


In [ ]:
more = 'In November Lake Dorn froze early, and Priya Nair extended her inflow model to it over the winter.'
update = doc.append(more)
print(update.new_chunks, 'new chunks,', update.prompts_sent, 'prompts sent')


## Load it again later

Retrieval needs no model. Pass `llm=` to answer questions or append.


In [ ]:
again = Saadi.load('saadi_state/aurora')
print(again)
print(again.retrieve('Who leads the Aurora team?', k=2)[0].text[:120])
